# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`


In [1]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 93.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take i

In [2]:
import os
import getpass
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


In [3]:
from markitdown import MarkItDown

DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return

    response = requests.get(url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")


for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")


⬇ Descargado: attention_paper.pdf (2.2 MB)
⬇ Descargado: rag_paper.pdf (0.9 MB)
✅ attention_paper.pdf → 40,174 caracteres
✅ rag_paper.pdf       → 69,121 caracteres

Inicio del paper de Attention:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog...


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)


Attention → 55 chunks
RAG       → 95 chunks
Total     → 150 chunks

Ejemplo de metadatos: {'title': 'Attention Is All You Need', 'source': 'attention_paper.pdf', 'chunk_index': 0}


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [5]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    truncate="END",
)
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = vectorstore.add_documents(all_docs)

print(f"✅ {len(ids)} chunks indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ 150 chunks indexados en Qdrant
✅ Dimensión de embeddings: 2048


---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [6]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=800,
)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts. "
        "If the excerpts do not contain enough evidence, say so explicitly. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = rag_chain.invoke(test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)


Pregunta: What is the Transformer architecture?
Respuesta: The Transformer is a sequence transduction architecture introduced in “Attention Is All You Need” that relies entirely on attention mechanisms, dispensing with recurrence and convolutions. It uses an encoder–decoder built from stacked self-attention and point-wise fully connected layers. The encoder is a stack of N = 6 identical layers; each layer has multi-head self-attention followed by a position-wise feed-forward network, with residual connections and layer normalization around each sub-layer, and model dimension d = 512. It draws global dependencies between input and output and allows significantly more parallelization than recurrent or convolutional architectures.


---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [7]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

Q: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?
A: 28.4 BLEU. This is reported in "Attention Is All You Need," which states that the Transformer (big) model established a new state-of-the-art BLEU score of 28.4 on the WMT 2014 English-to-German translation task.


<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [8]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

Q: How does RAG combine parametric and non-parametric memory?
A: RAG combines them in an end-to-end probabilistic model: the parametric memory is a pre-trained seq2seq transformer (BART), and the non-parametric memory is a dense vector index of Wikipedia accessed by a pre-trained neural retriever (DPR). Given an input, the retriever provides latent documents, and the generator conditions on both the input and those retrieved documents to produce the output. Latent documents are marginalized with a top-K approximation either per-output (same passages across the whole sequence) or per-token (different passages per token), and both the generator and retriever are jointly fine-tuned.

Paper: Retrieval-Augmented Generation (rag_paper.pdf)


<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [10]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

Q: What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?
A: The scaling factor \(1/\sqrt{d_k}\) counteracts the growth of dot-product magnitudes for large \(d_k\). Without scaling, large dot products push the softmax into regions with extremely small gradients. The paper notes that if query and key components are independent with mean 0 and variance 1, their dot product has mean 0 and variance \(d_k\), so scaling by \(\sqrt{d_k}\) keeps the values in a better-behaved range for softmax.

Paper: *Attention Is All You Need*


<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [11]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

Q: What are the two RAG model variants proposed in the paper and how do they differ?
A: The paper "Retrieval-Augmented Generation" (rag_paper.pdf) proposes two RAG variants:

1. **RAG-Sequence**: uses the same retrieved document to predict each target token (marginalizes latent documents on a per-output basis, assuming one document is responsible for all tokens).
2. **RAG-Token**: can predict each target token based on a different document (marginalizes on a per-token basis, where different documents are responsible for different tokens).


<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [12]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

Q: How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?
A: The base Transformer model uses **8 attention heads**, with **key dimension d_k = 64** and **value dimension d_v = 64** per head (since d_k = d_v = d_model/h = 512/8 = 64).

Source: *Attention Is All You Need* (attention_paper.pdf).


<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [13]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = rag_chain.invoke(q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

Q: The RAG paper uses FAISS for fast retrieval with an HNSW approximation. The Attention paper proposes the Transformer architecture. What seq2seq model does RAG use as its generator component, and is it based on the Transformer?

A: RAG uses **BART** as its generator component, and yes—it is based on the Transformer. The RAG paper describes BART as a “pre-trained seq2seq transformer” that conditions on the retrieved documents and the input to generate the output.  
Paper: *Retrieval-Augmented Generation* (rag_paper.pdf).

Fuentes recuperadas:
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Attention Is All You Need] — attention_paper.pdf


<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
- El modelo reconoció la limitación y abstuvo de inventar información gracias a las restricciones del *System Prompt*, previniendo alucinaciones.
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
- Sí, recuperó fragmentos de ambos artículos (`rag_paper.pdf` y `attention_paper.pdf`), lo que permitió conectar información entre dos fuentes distintas para construir una respuesta completa.
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
- Reducir `chunk_size` o aumentar `k` otorga contextos más específicos pero puede fragmentar ideas complejas.
- Aumentar `score_threshold` hace la búsqueda más estricta (evitando ruido), pero si es muy alto podría omitir información relevante.
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
- Permite optimizar costos y rendimiento: los embeddings requieren modelos livianos y rápidos optimizados para comparación vectorial, mientras que la generación requiere un LLM con gran capacidad de razonamiento.
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.
- **Persistencia:** Reemplazar Qdrant en memoria por una base de datos vectorial persistente (Qdrant Cloud/Cluster).
- **Observabilidad y Evaluación:** Integrar herramientas como LangSmith o Ragas para monitorear latencia y calidad de respuestas.
- **Escalabilidad y Seguridad:** Implementar control de acceso por roles (RBAC) para los documentos y mecanismos de caché para consultas frecuentes.


# **Reto C6**

In [18]:
import time
import os
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

qdrant_disk_path = "./qdrant_db"
lock_file_path = os.path.join(qdrant_disk_path, ".lock")

if os.path.exists(lock_file_path):
    try:
        os.remove(lock_file_path)
        print("🧹 Lock file removed successfully.")
    except Exception as e:
        print(f"⚠️ Could not remove lock file: {e}")

client_disk = QdrantClient(path=qdrant_disk_path)

if not client_disk.collection_exists("academic_papers_disk"):
    client_disk.create_collection(
        collection_name="academic_papers_disk",
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )

start_time = time.time()
vectorstore_disk = QdrantVectorStore(
    client=client_disk,
    collection_name="academic_papers_disk",
    embedding=embeddings,
)
vectorstore_disk.add_documents(all_docs)
index_time = time.time() - start_time

print(f"✅ Tiempo de creación del índice inicial: {index_time:.2f} segundos")

client_disk.close()

start_time = time.time()
client_reloaded = QdrantClient(path=qdrant_disk_path)
vectorstore_reloaded = QdrantVectorStore(
    client=client_reloaded,
    collection_name="academic_papers_disk",
    embedding=embeddings,
)
reload_time = time.time() - start_time

print(f"✅ Tiempo de recarga desde disco: {reload_time:.4f} segundos")
print(
    f"💡 La recarga desde disco es aproximadamente {index_time / reload_time:.1f} veces más rápida."
)

client_reloaded.close()

🧹 Lock file removed successfully.
✅ Tiempo de creación del índice inicial: 7.15 segundos
✅ Tiempo de recarga desde disco: 0.4191 segundos
💡 La recarga desde disco es aproximadamente 17.1 veces más rápida.


# **Reto C5**

In [19]:
retriever_sim = vectorstore.as_retriever(
    search_type="similarity", search_kwargs={"k": 5}
)

retriever_mmr = vectorstore.as_retriever(
    search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}
)

test_query = "What is the Transformer architecture and how does it compare to RNNs or RAG?"

docs_sim = retriever_sim.invoke(test_query)
docs_mmr = retriever_mmr.invoke(test_query)

print("=== 📌 1. Resultados con Similitud Pura ===")
for i, d in enumerate(docs_sim, 1):
  print(f"{i}. [{d.metadata['title']}] {d.page_content[:80]}...")

print("\n=== 📌 2. Resultados con MMR (Diversidad) ===")
for i, d in enumerate(docs_mmr, 1):
  print(f"{i}. [{d.metadata['title']}] {d.page_content[:80]}...")

=== 📌 1. Resultados con Similitud Pura ===
1. [Attention Is All You Need] Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethet...
2. [Attention Is All You Need] |                     | Dyeretal.              | (2016)[8]  |            |      ...
3. [Attention Is All You Need] mechanism. We propose a new simple network architecture, the Transformer,
baseds...
4. [Attention Is All You Need] entirelyonself-attentiontocomputerepresentationsofitsinputandoutputwithoutusings...
5. [Attention Is All You Need] attention,replacingtherecurrentlayersmostcommonlyusedinencoder-decoderarchitectu...

=== 📌 2. Resultados con MMR (Diversidad) ===
1. [Attention Is All You Need] Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethet...
2. [Retrieval-Augmented Generation] toprovideafinaloutput. Theseapproacheshaveprovedsuccessfulinanumberofdomainsincl...
3. [Attention Is All You Need] TheTransformerallowsforsignificantlymoreparallelizationandcanreachanew

# **Reto C3**

In [22]:
import gradio as gr
from qdrant_client.http import models as qmodels


def ask_rag_with_filter(question: str, selected_paper: str):
  question = (question or "").strip()
  if not question:
    return "Escribe una pregunta.", "No se realizaron búsquedas."

  qdrant_filter = None
  if selected_paper == "Attention Is All You Need":
    qdrant_filter = qmodels.Filter(
        must=[
            qmodels.FieldCondition(
                key="metadata.source",
                match=qmodels.MatchValue(value="attention_paper.pdf"),
            )
        ]
    )
  elif selected_paper == "Retrieval-Augmented Generation":
    qdrant_filter = qmodels.Filter(
        must=[
            qmodels.FieldCondition(
                key="metadata.source",
                match=qmodels.MatchValue(value="rag_paper.pdf"),
            )
        ]
    )

  retriever_filtered = vectorstore.as_retriever(
      search_type="similarity_score_threshold",
      search_kwargs={
          "k": 5,
          "score_threshold": 0.20,
          "filter": qdrant_filter,
      },
  )

  filtered_chain = (
      {
          "context": retriever_filtered | format_docs,
          "question": RunnablePassthrough(),
      }
      | prompt
      | llm
      | StrOutputParser()
  )

  try:
    answer = filtered_chain.invoke(question)
    docs = retriever_filtered.invoke(question)

    sources_info = "\n".join(
        f"- [{d.metadata.get('title')}] (archivo: {d.metadata.get('source')})"
        for d in docs
    )
    if not sources_info:
      sources_info = "No se encontraron documentos que cumplan el filtro."

  except Exception as e:
    answer = f"Error al procesar la consulta: {e}"
    sources_info = "Error en búsqueda."

  return answer, sources_info


with gr.Blocks(title="RAG con Filtro por Paper") as demo_c3:
  gr.Markdown(
      "# 📚 RAG Académico con Filtro por Metadata\nConsulta los papers con la"
      " opción de filtrar la búsqueda por un documento específico."
  )

  with gr.Row():
    with gr.Column():
      question_input = gr.Textbox(
          label="Pregunta",
          placeholder="Ej: What is the Transformer architecture?",
      )
      paper_dropdown = gr.Dropdown(
          choices=[
              "Todos",
              "Attention Is All You Need",
              "Retrieval-Augmented Generation",
          ],
          value="Todos",
          label="Filtrar por Documento",
      )
      btn_submit = gr.Button("Consultar", variant="primary")

    with gr.Column():
      answer_output = gr.Textbox(label="Respuesta del RAG", lines=6)
      sources_output = gr.Markdown(label="Fuentes Recuperadas")

  btn_submit.click(
      ask_rag_with_filter,
      inputs=[question_input, paper_dropdown],
      outputs=[answer_output, sources_output],
  )

demo_c3.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://91e675e0a14c0a1b83.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6ed7a00600090ba61b.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://91e675e0a14c0a1b83.gradio.live
